# Blood Work Analysis — a Two-Stage LLM Pipeline

This is my first real "application" built in this repo: paste in a blood
report, get back a plain-language health summary and a diet plan.

Two LLM calls, chained together:
1. **Stage 1 — Extraction:** read the raw report, pull out every test value, and flag each one
   HIGH / LOW / NORMAL against the reference ranges printed in the report.
2. **Stage 2 — Advice:** feed the *flagged* values (not the raw report) into a second prompt
   that writes a short summary and a practical diet plan.

Why two calls instead of one? Smaller, focused prompts are easier to get right, and splitting
"extract structured facts" from "give advice based on those facts" keeps each step's output a
lot more reliable than asking one giant prompt to do both at once.

See [`../../docs/07_workflow_vs_agent.md`](../../docs/07_workflow_vs_agent.md) for why this
counts as a **workflow**, not an agent — the LLM isn't deciding anything here, it's just
executing two fixed steps in order.

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## Load the sample report

In [ ]:
with open("sample_blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

## Stage 1 — Extract and flag abnormal values

Prompt engineering note: giving the model an explicit output format (`Test Name: value |
Status: ... | Reference: ...`) makes Stage 2's input far more consistent than letting the model
free-write its findings.

In [ ]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or
NORMAL based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

## Stage 2 — Health summary and diet plan

This prompt only ever sees `extracted_values` from Stage 1, never the raw report — a good
habit for keeping prompts focused and for not re-paying (in tokens) to re-process the same raw
text twice.

In [ ]:
diet_prompt = f"""
You are a nutritionist. Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines, in simple non-technical language
2. A short, practical diet plan with exactly two sections: (1) Foods to avoid (2) Foods to eat
   more of. Do not include any other sections.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

## Recap: the pipeline

```
blood_report (text)
      |
      v
[ LLM call 1: extraction prompt ]  -->  extracted_values (flagged HIGH/LOW/NORMAL)
      |
      v
[ LLM call 2: diet prompt ]        -->  health summary + diet plan
```

Next: [`streamlit_app/app.py`](streamlit_app/app.py) wraps this exact same two-stage logic in a
small web UI so it's actually usable without touching a notebook.